# Embed + CAGRA index on one GPU with Run:ai

One GPU runs llama-nemotron-embed and the OpenSearch CAGRA/HNSW builder together.
Run:ai's open-source [GPU Fractioning Operator](https://github.com/kai-scheduler/gpu-fractioning) gives each container a hard GPU-memory cap on driver r615 or newer. Compute is time-sliced.

The numbers live in `config/pipeline.yaml`. Change `embed_index_ratio` or `microbatch_size` there, then restart the GPU containers.

Workflow: embed `microbatch_size` vectors, build their CAGRA graph, repeat until all 1,000,000 vectors are indexed. Target Recall@10 is 0.95 against a cuVS brute-force cosine top-10.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "config" / "pipeline.yaml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "scripts"))

import pipeline_config

cfg = pipeline_config.load(ROOT / "config" / "pipeline.yaml")
share = pipeline_config.resolve(
    cfg,
    memory_mib=int(cfg["hardware"]["target_gpu_memory_mib"]),
    gpu_name=str(cfg["hardware"]["target_gpu"]),
)
pipeline_config.print_table(cfg, share)

## What the ratio means

`10:1` splits one GPU into 11 memory parts. Embedding gets 10 (request). Indexing gets 1 (request and limit).

`dynamic_embed: true` sets the embedding **limit** to the whole GPU. While the index builder is not allocating, that memory is available to embedding. The index cap does not grow.

On the target RTX PRO 6000 (96 GiB) that is:

| Workload | Request | Limit |
|---|---:|---:|
| Embedding | 89368 MiB | 98304 MiB |
| Indexing | 8936 MiB | 8936 MiB |

Hard caps need NVIDIA driver r615 / CUDA 13.4 or newer and container toolkit 1.20.1 or newer. `python scripts/pipeline_config.py` prints `driver_major` and `gpu_fraction_enforced` for the GPU it can see. Driver 610 leaves both containers uncapped. This repository does not install Run:ai; the README has the cluster install.

Re-render after a table edit:

```bash
python scripts/pipeline_config.py manifest --write deploy/k8s/embed-and-index.yaml
python scripts/pipeline_config.py manifest --detect --write deploy/k8s/embed-and-index.local.yaml
```

The first file matches the target GPU in the table. `--detect` uses GPU 0 on this machine.

## Docker on one machine

`scripts/start_gpu_share.sh` reads the table, exports the MiB caps, and restarts OpenSearch, the index builder, and NIM. Both GPU containers are pinned to GPU 0.

```bash
cp .env.example .env   # NGC_API_KEY, S3 bucket or LocalStack
./scripts/start_gpu_share.sh
./scripts/run_e2e.sh
```

`run_e2e.sh` embeds one microbatch, bulks it into OpenSearch, flushes so the remote builder builds CAGRA, then repeats. Embeddings are saved at `results/runai/embeddings_fp32.npy`.

Success metrics land in `results/runai/metrics.json`:

| Field | Spec |
|---|---|
| `embed_index_wall_s` | Embed + index time |
| `p95_latency_ms` | Search p95 |
| `avg_qps` | Search QPS |
| `recall_at_10` | Recall@10 vs cuVS brute force |

A smoke run is `./scripts/run_e2e.sh --limit 20000`.

## Kubernetes with Run:ai

Install [KAI Scheduler](https://github.com/NVIDIA/KAI-Scheduler) in NvFractions mode and the [GPU Fractioning Operator](https://github.com/kai-scheduler/gpu-fractioning) on a cluster whose GPU driver is r615 or newer (GPU Operator v26.7.1 with `driver.version=615.<patch>`, toolkit ≥ 1.20.1). The README has the helm commands. The checked-in manifest is the RTX PRO 6000 96 GiB split. On another GPU, apply `embed-and-index.local.yaml` from `manifest --detect` instead. OpenSearch is not in this manifest, and `run_e2e.sh` still needs ports 8000, 9200, and 1025 on the client machine.

Apply the manifest so the namespace exists, create the secrets, then restart the pods so they pick the secrets up.

```bash
kubectl apply -f deploy/k8s/embed-and-index.yaml
kubectl create secret generic ngc-api --from-literal=NGC_API_KEY="$NGC_API_KEY" -n embed-index
kubectl create secret generic s3-creds \
  --from-literal=AWS_REGION="$AWS_REGION" \
  --from-literal=AWS_ACCESS_KEY_ID="$AWS_ACCESS_KEY_ID" \
  --from-literal=AWS_SECRET_ACCESS_KEY="$AWS_SECRET_ACCESS_KEY" \
  --from-literal=S3_BUCKET="$S3_BUCKET" \
  -n embed-index
kubectl rollout restart deployment -n embed-index
```

Pod annotations are the Run:ai request:

- `gpu-fraction` is the share KAI Scheduler packs onto one GPU (`0.9091` and `0.0909`).
- `nvidia.com/container.<name>.gpu-memory.request` and `.limit` are what fractiond turns into `NVIDIA_GPU_MEMORY_REQUEST` and `NVIDIA_GPU_MEMORY_LIMIT`.

Neither pod requests `nvidia.com/gpu: 1`. That would take a whole GPU each. KAI places both fractions on the same device.

OpenSearch itself does not use the GPU. After it is up, set `knn.remote_index_build.service.endpoint` to `http://remote-index-builder.embed-index.svc:1025`.